# Refusal direction: all five notebooks in one run
Runs notebooks 01-05 end to end in a single Colab session, loading the model once. Uses the **`run-all` branch**, which contains the reference solutions for exercises 1-6, so every cell works without your own implementations. `main` still has the stubs if you want to do the exercises yourself.

Artifacts are written to `/content/refusal-direction/artifacts` (no Google Drive needed, since everything runs in one session). Download that folder at the end if you want to keep it.

Runtime: T4 GPU. Expect roughly 45-90 min, most of it candidate scoring (Part 1), generation (Parts 2-3) and GCG (Part 5).

In [ ]:
# --- Colab setup ---
# transformer_lens 2.18 needs numpy < 2. If Colab already loaded numpy 2, this cell restarts the
# runtime once after installing; then just run all cells again (the second pass is quick).
REPO = "https://github.com/aervale/refusal-direction"
BRANCH = "run-all"

import os
if not os.path.exists("/content/refusal-direction"):
    !git clone -q -b $BRANCH $REPO /content/refusal-direction
%cd /content/refusal-direction
!git pull -q
# pyarrow >= 22 refuses to import with numpy 1.x, which transformer_lens 2.18 requires
%pip install -q -e . "pyarrow<22"

import numpy
if int(numpy.__version__.split(".")[0]) >= 2:
    print(f"numpy {numpy.__version__} is loaded; restarting the runtime. Run all cells again afterwards.")
    os.kill(os.getpid(), 9)
print("numpy", numpy.__version__, "- ready")

In [ ]:
%load_ext autoreload
%autoreload 2
import gc, torch, pandas as pd, matplotlib.pyplot as plt
from refusal_direction.config import Config
cfg = Config(artifact_dir="artifacts")
print(cfg)

In [ ]:
from refusal_direction.model import load_model, tokenize_instructions, eoi_positions, generate, last_token_logits
from refusal_direction.data import load_datasets
from refusal_direction.refusal import refusal_token_ids, refusal_score, refusal_rate

model = load_model(cfg)
data = load_datasets(cfg)
tok = lambda instrs, suffix="": tokenize_instructions(model, instrs, cfg.system_prompt, suffix)
refusal_toks = refusal_token_ids(model.tokenizer)
print(model.cfg.n_layers, "layers | d_model", model.cfg.d_model, "| refusal tokens", refusal_toks,
      [model.tokenizer.decode([t]) for t in refusal_toks])

# Part 1: Find the refusal direction
**Paper:** Sections 2.2-2.3, Appendix A & C. **Exercises used:** 1, 2, 3, 4 (run `pytest` locally first).

Pipeline: base-model sanity check -> filter data -> difference-in-means at every (position, layer) -> score every candidate -> pick one -> save it.

## Sanity check: does TransformerLens load Qwen2.5 correctly? (PROJECT.md assumption 11)
Compare last-position logits from TransformerLens and Hugging Face, **both in float32**, on 5 prompts, once as a left-padded batch and once per prompt with no padding.
- **Prediction:** max |diff| under about 1e-3 and the same top-1 token for all 5 prompts.
- **Bad in both columns:** a loading / weight-conversion bug.
- **Bad only when batched:** a padding bug (attention mask or rotary positions).

The float32 TL copy is loaded fresh, because casting the fp16 working model up wouldn't recover the original bf16 checkpoint weights. The last line also reports the fp16 working model against the same reference, to show how much difference precision alone adds.

In [ ]:
import gc, dataclasses
from transformers import AutoModelForCausalLM

check_prompts = data["harmful_val"][:2] + data["harmless_val"][:3]
check_kinds = ["harmful"] * 2 + ["harmless"] * 3
# The same token ids go to every model: one left-padded batch, then each prompt alone (no padding).
check_inputs = [tok(check_prompts)] + [tok([p]) for p in check_prompts]

def tl_logits(m):
    return [last_token_logits(m, *x).cpu() for x in check_inputs]

@torch.no_grad()
def hf_logits(m):
    out = []
    for toks, mask in check_inputs:
        pos = (mask.cumsum(-1) - 1).clamp(min=0)  # explicit positions so HF handles left padding correctly
        out.append(m(input_ids=toks, attention_mask=mask, position_ids=pos).logits[:, -1, :].float().cpu())
    return out

# One float32 model on the GPU at a time, next to the fp16 working model.
tl32 = load_model(dataclasses.replace(cfg, dtype="float32"))
tl32_out = tl_logits(tl32)
del tl32; gc.collect(); torch.cuda.empty_cache()
hf = AutoModelForCausalLM.from_pretrained(cfg.model_name, dtype=torch.float32).to(model.cfg.device).eval()
hf_out = hf_logits(hf)
del hf; gc.collect(); torch.cuda.empty_cache()
tl16_out = tl_logits(model)

def compare(a, b):
    """Per-prompt max |logit diff| and top-1 agreement, for the padded batch and for each prompt alone."""
    return {
        "padded_max_diff": (a[0] - b[0]).abs().max(-1).values.numpy(),
        "padded_top1_agree": (a[0].argmax(-1) == b[0].argmax(-1)).numpy(),
        "alone_max_diff": [float((x - y).abs().max()) for x, y in zip(a[1:], b[1:])],
        "alone_top1_agree": [bool(x.argmax(-1)[0] == y.argmax(-1)[0]) for x, y in zip(a[1:], b[1:])],
    }

check = pd.DataFrame({"kind": check_kinds, "prompt": [p[:50] for p in check_prompts], **compare(tl32_out, hf_out)})
print(f"padded batch length {check_inputs[0][0].shape[1]} | prediction: max |diff| < 1e-3, top-1 5/5")
print(f"TL float32, left-padded batch: max |diff| {check.padded_max_diff.max():.2e}, top-1 {check.padded_top1_agree.sum()}/{len(check)}")
print(f"TL float32, each prompt alone: max |diff| {max(check.alone_max_diff):.2e}, top-1 {sum(check.alone_top1_agree)}/{len(check)}")
c16 = compare(tl16_out, hf_out)
print(f"TL {model.cfg.dtype} (working model), padded batch: max |diff| {c16['padded_max_diff'].max():.2e}, top-1 {c16['padded_top1_agree'].sum()}/{len(check)}")
display(check)

## Sanity check: does the base model refuse what it should?

In [ ]:
sample_h, sample_hl = data["harmful_val"][:4], data["harmless_val"][:4]
outs_h = generate(model, *tok(sample_h), max_new_tokens=48)
outs_hl = generate(model, *tok(sample_hl), max_new_tokens=48)
for instr, out in list(zip(*(sample_h, outs_h)))[:4]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)
for instr, out in list(zip(*(sample_hl, outs_hl)))[:4]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)

**Check:** do refusals actually start with the refusal tokens printed above? If Qwen starts its refusals with some other token, `refusal_score` is measuring the wrong thing - fix `refusal_token_ids(..., starts=...)` before going on.

## Filter the data (App. A)

In [ ]:
from refusal_direction.selection import filter_by_refusal
if cfg.filter_train:
    for split, n in [("train", cfg.n_train), ("val", cfg.n_val)]:
        data[f"harmful_{split}"] = filter_by_refusal(model, data[f"harmful_{split}"], tok, refusal_toks, True, n, cfg.batch_size)
        data[f"harmless_{split}"] = filter_by_refusal(model, data[f"harmless_{split}"], tok, refusal_toks, False, n, cfg.batch_size)
print({k: len(v) for k, v in data.items()})

## Candidate directions: difference-in-means (Exercise 3)

In [ ]:
from refusal_direction.directions import get_candidate_directions
positions = eoi_positions(model)
print("post-instruction tokens:", [model.tokenizer.decode([t]) for t in tok(["x"])[0][0, positions]])
candidates = get_candidate_directions(model, tok(data["harmful_train"]), tok(data["harmless_train"]), positions, cfg.batch_size)
candidates.shape  # [n_pos, n_layers, d_model]

In [ ]:
# How big is the harmful/harmless gap, by depth? (norm of each candidate)
plt.figure(figsize=(8, 3))
for i, p in enumerate(positions):
    plt.plot(candidates[i].norm(dim=-1).cpu(), label=f"pos {p}")
plt.xlabel("layer"); plt.ylabel("||r||"); plt.legend(); plt.title("Difference-in-means norm"); plt.show()

## Score every candidate (takes a few minutes on a T4)

In [ ]:
from refusal_direction.selection import score_candidates
from refusal_direction import artifacts
scores = score_candidates(model, candidates, positions, tok(data["harmful_val"]), tok(data["harmless_val"]), refusal_toks, cfg.batch_size)
scores.to_csv(artifacts.path(cfg, "candidate_scores.csv"), index=False)
scores.sort_values("bypass_score").head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for metric, ax in zip(["bypass_score", "induce_score", "kl_score"], axes):
    for p, g in scores.groupby("pos"):
        ax.plot(g.layer, g[metric], label=f"pos {p}")
    ax.set_title(metric); ax.set_xlabel("layer")
axes[0].axhline(0, c="k", lw=.5); axes[1].axhline(cfg.induce_threshold, c="r", ls="--")
axes[2].axhline(cfg.kl_threshold, c="r", ls="--"); axes[2].set_yscale("log")
for ax in axes: ax.axvline(cfg.prune_layer_fraction * model.cfg.n_layers, c="gray", ls=":")
axes[0].legend(); plt.show()

## Pick the direction (Exercise 4) and save it

In [ ]:
from refusal_direction.selection import select_direction
best = select_direction(scores, model.cfg.n_layers, cfg)
direction = candidates[int(best.pos_idx), int(best.layer)]
meta = {k: (float(v) if k.endswith("score") else int(v)) for k, v in best.items()}
print(meta)
artifacts.save_direction(cfg, direction, meta)

## Questions to write up
1. Which layer/position won? Paper: Qwen 1.8B picked a middle layer at the last eoi token - same for you?
2. Plot: is there a band of layers where *many* candidates work, or one sharp winner? What does that suggest about how localized refusal is?
3. Why does the paper prune layers >= 0.8L? Disable the filter and see what gets picked - then generate with it.

# Part 2: Ablate and add the direction
**Paper:** Section 3 (Figures 1 & 3). Bypass refusal on harmful prompts by ablating `r`; induce refusal on harmless prompts by adding it.
Includes a **random-direction control**, which the paper doesn't show but you should: it tells you whether *any* ablation breaks refusal.

In [ ]:
from refusal_direction import artifacts
from refusal_direction.interventions import get_ablation_hooks, get_addition_hooks
direction, meta = artifacts.load_direction(cfg)
direction = direction.to(model.cfg.device)
layer = meta["layer"]; print(meta)
torch.manual_seed(cfg.seed)
random_dir = torch.randn_like(direction)  # control; scale is irrelevant for ablation

## Harmful prompts: baseline vs. ablation vs. random-direction ablation

In [ ]:
harm = data["harmful_test"]
H = tok(harm)
results = {}
gens = {}
for name, hooks in [("baseline", []), ("ablate_r", get_ablation_hooks(model, direction)),
                    ("ablate_random", get_ablation_hooks(model, random_dir)),
                    ("subtract_r_at_layer", get_addition_hooks(model, direction, layer, coeff=-1.0))]:
    gens[name] = generate(model, *H, fwd_hooks=hooks, max_new_tokens=cfg.max_new_tokens, batch_size=cfg.batch_size)
    results[f"harmful/{name}"] = refusal_rate(gens[name])
results

In [ ]:
for instr, out in list(zip(*(harm, gens["ablate_r"])))[:3]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)

## Harmless prompts: baseline vs. adding `r` at its layer

In [ ]:
safe = data["harmless_test"]
S = tok(safe)
for name, hooks in [("baseline", []), ("add_r", get_addition_hooks(model, direction, layer))]:
    gens["harmless_" + name] = generate(model, *S, fwd_hooks=hooks, max_new_tokens=cfg.max_new_tokens, batch_size=cfg.batch_size)
    results[f"harmless/{name}"] = refusal_rate(gens["harmless_" + name])
artifacts.save_json(cfg, "refusal_rates.json", results)
results

In [ ]:
for instr, out in list(zip(*(safe, gens["harmless_add_r"])))[:3]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)

In [ ]:
pd.Series(results).plot.barh(figsize=(7, 3), xlim=(0, 1), title="substring refusal rate"); plt.show()

## Questions to write up
1. Compare to the paper's Figure 1. Is the bypass as clean on Qwen2.5-1.5B? Does the random control stay near baseline?
2. Read ~10 ablated completions by hand. Substring matching counts "Sure, here's ... I can't stress enough how illegal this is" as a refusal and vague non-answers as compliance. How far off is the metric? (This is why the paper also uses a safety classifier.)
3. Sweep `coeff` in `get_addition_hooks` (0.5, 1, 2, 4). Where does induced refusal saturate, and when does the output turn to gibberish?

# Part 3: Bake the ablation into the weights
**Paper:** Section 4. **Exercise used:** 5.
If nothing writes along `r_hat`, the residual stream never contains it. Editing ~2 matrices per layer gives a model that doesn't refuse, with no hooks.

In [ ]:
import copy
from refusal_direction import artifacts
from refusal_direction.interventions import get_ablation_hooks
from refusal_direction.orthogonalize import orthogonalize_weights
direction, meta = artifacts.load_direction(cfg)
direction = direction.to(model.cfg.device)

edited = copy.deepcopy(model)          # ~3 GB extra in fp16; fits on a T4
orthogonalize_weights(edited, direction)

## Equivalence check: edited model vs. hook-ablated model

In [ ]:
H = tok(data["harmful_test"][:16])
hooked = last_token_logits(model, *H, fwd_hooks=get_ablation_hooks(model, direction))
edit = last_token_logits(edited, *H)
print("max |logit diff|:", (hooked - edit).abs().max().item())
print("top-1 agreement:", (hooked.argmax(-1) == edit.argmax(-1)).float().mean().item())
# Not exactly 0 in fp16/bf16: the two compute the same function in a different order.

In [ ]:
harm = data["harmful_test"]
outs = generate(edited, *tok(harm), max_new_tokens=cfg.max_new_tokens, batch_size=cfg.batch_size)
rate = refusal_rate(outs)
artifacts.save_json(cfg, "orthogonalized_refusal_rate.json", {"harmful/orthogonalized": rate})
print("refusal rate (orthogonalized):", rate)

In [ ]:
for instr, out in list(zip(*(harm, outs)))[:3]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)

## Questions to write up
1. How many parameters did you change, as a fraction of the model? What does it mean that a rank-1 edit removes a safety behavior?
2. The edited model's weights could be exported and shared. What does this imply for open-weight release policy? (Paper, Section 7 / ethics statement.)
3. Stretch: write a function that copies the edited TransformerLens weights back into a HF `AutoModelForCausalLM` and save it locally.

In [ ]:
del edited; gc.collect(); torch.cuda.empty_cache()

# Part 4: Did we break the model?
**Paper:** Section 4.2 (Table 1). The claim: the orthogonalized model loses refusal but keeps most capability.
Light version: WikiText CE, MMLU subset, and KL on harmless prompts.

In [ ]:
import copy, torch.nn.functional as F
from refusal_direction import artifacts
from refusal_direction.orthogonalize import orthogonalize_weights
from refusal_direction.capability import load_wikitext, mean_cross_entropy, load_mmlu, mmlu_accuracy
direction, meta = artifacts.load_direction(cfg)
edited = copy.deepcopy(model); orthogonalize_weights(edited, direction.to(model.cfg.device))

# control: orthogonalize a random direction of the same norm
torch.manual_seed(cfg.seed)
rand_edited = copy.deepcopy(model); orthogonalize_weights(rand_edited, torch.randn_like(direction).to(model.cfg.device))

In [ ]:
wiki = load_wikitext(n=200)
mmlu = load_mmlu(n=200)
S = tok(data["harmless_test"])
base_lp = last_token_logits(model, *S).log_softmax(-1)
rows = {}
for name, m in [("base", model), ("orthogonalized", edited), ("random_orth", rand_edited)]:
    lp = last_token_logits(m, *S).log_softmax(-1)
    rows[name] = {
        "wikitext_CE": mean_cross_entropy(m, wiki),
        "mmlu_acc": mmlu_accuracy(m, mmlu, cfg.system_prompt),
        "harmless_KL_vs_base": F.kl_div(lp, base_lp, log_target=True, reduction="batchmean").item(),
    }
table = pd.DataFrame(rows).T
artifacts.save_json(cfg, "capability.json", rows)
table

## Questions to write up
1. Compare the CE / MMLU deltas to Table 1 for Qwen 1.8B. Bigger or smaller drop?
2. 200 MMLU questions gives a standard error of about 3.5 points. Is your difference significant? (Run a paired bootstrap if you want to be sure.)
3. The paper notes TruthfulQA drops noticeably. Why might removing refusal hurt truthfulness specifically?

In [ ]:
del edited, rand_edited; gc.collect(); torch.cuda.empty_cache()

# Part 5: Mechanistic extras
**Paper:** Section 5 (adversarial suffixes) + probes. **Exercise used:** 6.
1. Where in depth does the direction separate harmful from harmless?
2. What does the direction read as through the unembedding?
3. Which attention heads write it?
4. Does an adversarial suffix suppress it, and do those heads look at the suffix instead of the instruction?

In [ ]:
from refusal_direction import artifacts
from refusal_direction.mechanistic import projection_by_layer, head_contributions, attention_from_position, logit_lens_direction
direction, meta = artifacts.load_direction(cfg)
direction = direction.to(model.cfg.device)
H, S = tok(data["harmful_val"]), tok(data["harmless_val"])

## 1. Projection onto r across layers (last token)

In [ ]:
ph = projection_by_layer(model, *H, direction).cpu()
ps = projection_by_layer(model, *S, direction).cpu()
plt.figure(figsize=(8, 3))
for p, lab in [(ph, "harmful"), (ps, "harmless")]:
    plt.plot(p.mean(0), label=lab); plt.fill_between(range(p.shape[1]), p.mean(0) - p.std(0), p.mean(0) + p.std(0), alpha=.2)
plt.axvline(meta["layer"], c="gray", ls=":"); plt.xlabel("layer"); plt.ylabel("x . r_hat"); plt.legend(); plt.show()

## 2. Logit lens on r

In [ ]:
ll = logit_lens_direction(model, direction, k=15)
pd.DataFrame({"top": [t for t, _ in ll["top"]], "bottom": [t for t, _ in ll["bottom"]]})

## 3. Which heads write the direction?

In [ ]:
hc = head_contributions(model, *H, direction).mean(0).cpu()   # [n_layers, n_heads]
plt.figure(figsize=(6, 6)); plt.imshow(hc, cmap="RdBu", vmin=-hc.abs().max(), vmax=hc.abs().max())
plt.colorbar(); plt.xlabel("head"); plt.ylabel("layer"); plt.title("mean head output . r_hat (harmful)"); plt.show()
flat = hc.flatten().topk(8)
top_heads = [(int(i) // model.cfg.n_heads, int(i) % model.cfg.n_heads) for i in flat.indices]
top_heads

## 4. Adversarial suffix (paper Section 5)
You need a suffix that jailbreaks *your* model. Option A (recommended): optimize one with GCG via `nanogcg` (~10-20 min on a T4). Option B: skip to the random-suffix control to get the plumbing working first.

The paper optimizes one universal suffix over many prompts; nanoGCG optimizes per prompt, which is fine for a first look.

In [ ]:
# Option A: GCG. Uses a separate HF copy of the model (nanogcg doesn't speak TransformerLens).
%pip install -q nanogcg
import nanogcg
from transformers import AutoModelForCausalLM, AutoTokenizer
hf_tok = AutoTokenizer.from_pretrained(cfg.model_name)
# Match OUR prompt format exactly (no auto-injected Qwen system prompt):
hf_tok.chat_template = ("{% for m in messages %}<|im_start|>{{ m['role'] }}\n{{ m['content'] }}<|im_end|>\n{% endfor %}"
                        "{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}")
hf_model = AutoModelForCausalLM.from_pretrained(cfg.model_name, torch_dtype=model.cfg.dtype).to(model.cfg.device)
target_instr = data["harmful_val"][0]
gcg_cfg = nanogcg.GCGConfig(num_steps=250, search_width=256, topk=256, seed=cfg.seed, verbosity="WARNING")
res = nanogcg.run(hf_model, hf_tok, target_instr + " {optim_str}", "Sure, here", gcg_cfg)
SUFFIX = " " + res.best_string
del hf_model; torch.cuda.empty_cache()
print(repr(SUFFIX))

In [ ]:
# Option B / control: random tokens of similar length
g = torch.Generator().manual_seed(0)
RANDOM_SUFFIX = " " + model.tokenizer.decode(torch.randint(1000, 30000, (20,), generator=g))
SUFFIX = globals().get("SUFFIX", RANDOM_SUFFIX)

In [ ]:
instrs = [target_instr] if "target_instr" in globals() else data["harmful_val"][:8]
conds = {"harmful": "", "harmful+adv_suffix": SUFFIX, "harmful+random_suffix": RANDOM_SUFFIX}
for name, sfx in conds.items():
    T = tok(instrs, sfx)
    out = generate(model, *T, max_new_tokens=40, show_progress=False)[0]
    proj = projection_by_layer(model, *T, direction)[:, meta["layer"]].mean().item()
    print(f"{name:24s} proj@layer{meta['layer']}={proj:7.2f} | {out[:100]!r}")

In [ ]:
# Where do the top heads look? Attention mass from the last token on the instruction vs. the suffix.
n_eoi, n_sfx = len(eoi_positions(model)), len(model.tokenizer.encode(SUFFIX, add_special_tokens=False))
T = tok(instrs, SUFFIX)
pat = attention_from_position(model, *T, top_heads)  # [batch, heads, key_pos]
seq = T[0].shape[1]
sfx_slice = slice(seq - n_eoi - n_sfx, seq - n_eoi)   # approx: token boundaries can shift by one
instr_slice = slice(0, seq - n_eoi - n_sfx)
pd.DataFrame({"head": [f"L{l}H{h}" for l, h in top_heads],
              "attn_on_instruction": pat[..., instr_slice].sum(-1).mean(0).cpu(),
              "attn_on_suffix": pat[..., sfx_slice].sum(-1).mean(0).cpu()})
# Compare with the same table computed with RANDOM_SUFFIX.

## Questions to write up
1. At which layer does harmful/harmless separation appear? Is it before the selected layer?
2. Does the logit lens show refusal-flavoured tokens, or is r mostly not a direct "say I'm sorry" direction? What does that suggest about how refusal is implemented downstream?
3. With the adversarial suffix: does the projection drop, and do the top heads shift attention onto the suffix (paper Fig. 5)? Does the random suffix do the same? If not, what does the suffix do that random tokens don't?